<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/learning_functions_17_01_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

# download the names.txt file from github
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

--2026-01-17 23:44:15--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt.5’

names.txt.5         100%[===================>] 222.80K  --.-KB/s    in 0.03s   

2026-01-17 23:44:15 (7.48 MB/s) - ‘names.txt.5’ saved [228145/228145]



['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [41]:
class Linear:

  def __init__(self, fan_in, fan_out, bias=True):
    self.weight=torch.randn(fan_in,fan_out)/ (fan_in**0.5)
    self.bias = torch.zeros(1,fan_out) if bias  else None
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])


In [42]:
class BatchNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps=eps #Maybe tensor so that we affect grads of n above
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    #
    self.run_mean=torch.zeros(dim)
    self.run_var=torch.ones(dim)
  def __call__(self, x):
     if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
     else:
      xmean = self.run_mean
      xvar = self.run_var

     self.out= self.gamma*((x-xmean)/torch.sqrt(xvar+self.eps))+self.beta
     if self.training:
      with torch.no_grad():
        self.run_mean = (1 - self.momentum) * self.run_mean + self.momentum * xmean
        self.run_var = (1 - self.momentum) * self.run_var + self.momentum * xvar

  def parameters(self):
    return [self.gamma,self.beta]

In [43]:
class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []


In [59]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP

C = torch.randn((27, n_embd))
layers = [
  Linear(n_embd * 3, n_hidden, bias=False), #BatchNorm1d(n_hidden), #Tanh(),
  Linear(           n_hidden, 27, bias=False)#, #BatchNorm1d(27),
]


In [63]:

with torch.no_grad():
  # last layer: make less confident
  #layers[-1].gamma *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

6024


In [64]:

chars= sorted(set(".".join(words)))
itos={i:s for i,s in enumerate(chars)}
stoi={s:i for i,s in enumerate(chars)}
def data_prep(data,context):
  data= ["."*context +word +"." for word in data]
  Y=[stoi[i] for ch in data for i in ch[context:]]
  X=[word[i:i+context] for word in data for i in range(len(word)-context)]
  X=[stoi[i] for x in X for i in x]
  X=torch.tensor(X).view(-1,context)
  Y=torch.tensor(Y)
  return X, Y
context=3
Xtr,Ytr=data_prep(words,context)

In [65]:
# same optimization as last time
max_steps = 2000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function

  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization

TypeError: cross_entropy_loss(): argument 'input' (position 1) must be Tensor, not NoneType

In [34]:
print(layers[1](layers[0](x)))

None


In [33]:
layers[0](x)

tensor([[ 0.4924, -0.3998, -1.6805,  ...,  1.3549,  1.5410,  0.2951],
        [ 0.5189,  2.8903, -0.2427,  ...,  0.3193,  0.3413,  0.1237],
        [-2.0014,  0.1425, -1.1270,  ..., -0.7669,  0.5225, -2.2212],
        ...,
        [ 0.9414,  0.3446, -0.1470,  ...,  0.0961,  0.3308, -0.1806],
        [ 0.1249,  1.7696,  0.3592,  ..., -0.0763, -0.2445, -0.2613],
        [ 0.3310, -1.0849, -0.2810,  ..., -1.0080,  1.5136, -0.9734]],
       grad_fn=<MmBackward0>)

In [ ]:
z=layers[2](emb.view(emb.shape[0], -1))

In [ ]:
Yb.size()

In [ ]:
F.cross_entropy(z,Yb)